In [ ]:
import requests

url = (
    "https://usdmdataservices.unl.edu/api/StateStatistics/GetDSCI"
    "?aoi=25"
    "&startdate=1/4/2000"
    "&enddate=4/9/2026"
    "&statisticsType=1"
)

response = requests.get(url, headers={"Accept": "text/csv"}, timeout=60)
response.raise_for_status()

with open("massachusetts_dsci.csv", "wb") as f:
    f.write(response.content)

print("Saved as massachusetts_dsci.csv")

Saved as massachusetts_dsci.csv


In [ ]:
import pandas as pd

df = pd.read_csv("massachusetts_dsci.csv")

# basic check
print(df.head())
print("\nColumns:", df.columns)
print("\nShape:", df.shape)

            Name   MapDate  DSCI
0  Massachusetts  20000104     0
1  Massachusetts  20000111     0
2  Massachusetts  20000118     0
3  Massachusetts  20000125     0
4  Massachusetts  20000201     0

Columns: Index(['Name', 'MapDate', 'DSCI'], dtype='object')

Shape: (1371, 3)


In [ ]:
# Convert MapDate to datetime
df['MapDate'] = pd.to_datetime(df['MapDate'], format='%Y%m%d')

# Sort by date (important for time series)
df = df.sort_values('MapDate').reset_index(drop=True)

# Check again
print(df.head())
print(df.tail())

            Name    MapDate  DSCI
0  Massachusetts 2000-01-04     0
1  Massachusetts 2000-01-11     0
2  Massachusetts 2000-01-18     0
3  Massachusetts 2000-01-25     0
4  Massachusetts 2000-02-01     0
               Name    MapDate  DSCI
1366  Massachusetts 2026-03-10   164
1367  Massachusetts 2026-03-17   146
1368  Massachusetts 2026-03-24   131
1369  Massachusetts 2026-03-31   122
1370  Massachusetts 2026-04-07   111


In [ ]:
# Training data: 2011–2020
train_df = df[(df['MapDate'] >= '2011-01-01') & (df['MapDate'] <= '2020-12-31')].reset_index(drop=True)

# Testing data: 2021
test_df = df[(df['MapDate'] >= '2021-01-01') & (df['MapDate'] <= '2021-12-31')].reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 522
Test size: 52


In [ ]:
def classify(dsci):
    if dsci < 24:
        return 0   # near normal
    elif dsci < 116:
        return 1   # mild/moderate drought
    else:
        return 2   # high drought

train_df['State'] = train_df['DSCI'].apply(classify)
test_df['State'] = test_df['DSCI'].apply(classify)

print("Train sample:")
print(train_df[['MapDate', 'DSCI', 'State']].head())

print("\nTest sample:")
print(test_df[['MapDate', 'DSCI', 'State']].head())

Train sample:
     MapDate  DSCI  State
0 2011-01-04     0      0
1 2011-01-11     0      0
2 2011-01-18     0      0
3 2011-01-25     0      0
4 2011-02-01     0      0

Test sample:
     MapDate  DSCI  State
0 2021-01-05     0      0
1 2021-01-12     0      0
2 2021-01-19     0      0
3 2021-01-26     0      0
4 2021-02-02     0      0


In [ ]:
import numpy as np
import pandas as pd

# 3 states: 0, 1, 2
N = np.zeros((3, 3), dtype=int)

for i in range(len(train_df) - 1):
    current_state = train_df.loc[i, 'State']
    next_state = train_df.loc[i + 1, 'State']
    N[current_state, next_state] += 1

N_df = pd.DataFrame(
    N,
    index=['Normal', 'Moderate', 'Severe'],
    columns=['Normal', 'Moderate', 'Severe']
)

print("Transition count matrix N_ij:")
print(N_df)

Transition count matrix N_ij:
          Normal  Moderate  Severe
Normal       297        12       0
Moderate      11        83      12
Severe         1        11      94


In [ ]:
alpha = 0.5
num_states = 3

# Apply formula: P_ij = (N_ij + alpha) / sum_j(N_ij + alpha)
P = (N + alpha) / (N.sum(axis=1, keepdims=True) + alpha * num_states)

# Convert to DataFrame for clarity
P_df = pd.DataFrame(
    P,
    index=['Normal', 'Moderate', 'Severe'],
    columns=['Normal', 'Moderate', 'Severe']
)

print("Smoothed Transition Probability Matrix P_ij:")
print(P_df)

print("\nRow sums (should be 1):")
print(P_df.sum(axis=1))

Smoothed Transition Probability Matrix P_ij:
            Normal  Moderate    Severe
Normal    0.958132  0.040258  0.001610
Moderate  0.106977  0.776744  0.116279
Severe    0.013953  0.106977  0.879070

Row sums (should be 1):
Normal      1.0
Moderate    1.0
Severe      1.0
dtype: float64


In [ ]:
y_true = []
y_pred = []
y_prob = []

for i in range(len(test_df) - 1):
    current_state = test_df.loc[i, 'State']
    true_next_state = test_df.loc[i + 1, 'State']

    # Get probability distribution from trained matrix
    prob_next = P[current_state]

    # Predicted state = max probability
    pred_next_state = np.argmax(prob_next)

    y_true.append(true_next_state)
    y_pred.append(pred_next_state)
    y_prob.append(prob_next)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("First 10 true states:     ", y_true[:10])
print("First 10 predicted states:", y_pred[:10])
print("Total predictions:", len(y_true))

First 10 true states:      [0 0 0 0 0 0 0 0 0 1]
First 10 predicted states: [0 0 0 0 0 0 0 0 0 0]
Total predictions: 51


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')

print("Accuracy:", accuracy)
print("F1-macro:", f1_macro)

Accuracy: 0.9607843137254902
F1-macro: 0.9258720930232558


In [ ]:
from sklearn.metrics import log_loss

# Log Loss (cross-entropy)
logloss = log_loss(y_true, y_prob, labels=[0,1,2])

# Brier Score (multiclass)
y_true_onehot = np.eye(3)[y_true]
brier_score = np.mean(np.sum((y_prob - y_true_onehot) ** 2, axis=1))

print("Log Loss:", logloss)
print("Brier Score:", brier_score)

Log Loss: 0.17671441152019213
Brier Score: 0.07684165064751795
